# Sequence modeling and alternative loss boundaries

The **loss** is one important recipe boundary: ordinary softmax
cross-entropy uses exponential and division operations, while a perceptron
gradient and a lookup/shift approximation expose different numerical and
hardware trade-offs.

TikiTorch exposes ordinary host cross-entropy, a perceptron boundary, and an
experimental lookup/shift approximation. This notebook compares those choices
on a character LM, then shows `TropicalLion` and the streaming `MorphSSM` cell.
It is a pedagogical run, not a silicon or publication benchmark.


In [ ]:
# --- Setup: install TikiTorch (run once) ---
# TikiTorch is research code, not on PyPI. In Colab, install from the repo.
try:
    import tikitorch  # already installed?
except ImportError:
    REPO_URL = "https://github.com/nyuolab/tikitorch.git"
    import subprocess, sys, os
    if not os.path.exists("tikitorch_repo"):
        subprocess.run(
            ["git", "clone", "--branch", "pc-morphssm", "--single-branch",
             REPO_URL, "tikitorch_repo"],
            check=True,
        )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-e", "tikitorch_repo"],
        check=True,
    )
    import tikitorch
import torch
print("TikiTorch", getattr(tikitorch, "__version__", "?"), "| torch", torch.__version__)


## A character-level Shakespeare LM at the silicon dims (D=128, V=128)


In [ ]:
import torch, tikitorch as tt, tikitorch.optim as optim
V = 128

# Tiny built-in text so the notebook is self-contained; swap in any corpus.
import urllib.request, os
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
try:
    text = urllib.request.urlopen(url, timeout=10).read().decode("utf-8", "replace")
except Exception:
    text = ("To be, or not to be, that is the question:\n" * 2000)   # fallback
ids = torch.tensor([min(ord(c), V - 1) for c in text], dtype=torch.long)
print("corpus chars:", ids.numel())

def batch(bs, L, gen):
    s = torch.randint(0, ids.numel() - L - 1, (bs,), generator=gen)
    x = torch.stack([ids[i:i+L] for i in s]); y = torch.stack([ids[i+1:i+L+1] for i in s])
    return x, y


## Compare perceptron, host CE, and lookup/shift conditioning

We hold the model, optimizer, seed, and data fixed and vary **only the loss
conditioning**: the perceptron baseline, the host-softmax `tropical_ce`, and the
experimental `tropical_ce_lut`. The comparison describes this executed host
configuration only; it does not establish a compiled full-step result.


In [ ]:
def run(loss_type, steps=400, L=48, bs=24, seed=0):
    torch.manual_seed(seed); gen = torch.Generator().manual_seed(seed)
    m = tt.models.TropicalTransformer(d_model=128, n_heads=4, n_layers=1, d_ff=512,
                                      tau=0.5, max_seq_len=L, task_type="lm", vocab_size=V)
    m.loss_type = loss_type
    opt = optim.SignSGD(m.parameters(), lr=2e-3, momentum=0.9)
    for _ in range(steps):
        x, y = batch(bs, L, gen); opt.zero_grad()
        m.compute_loss(x, y).backward(); opt.step()
    gen2 = torch.Generator().manual_seed(seed + 1)
    accs = []
    with torch.no_grad():
        for _ in range(8):
            x, y = batch(bs, L, gen2)
            accs.append((m(x).reshape(-1, V).argmax(-1) == y.reshape(-1)).float().mean().item())
    return sum(accs) / len(accs)

rows = [("perceptron (silicon baseline)", "perceptron"),
        ("tropical_ce (host softmax)", "tropical_ce"),
        ("tropical_ce_lut (lookup/shift)", "tropical_ce_lut"),
        ("cross_entropy (float ceiling)", "cross_entropy")]
res = {lt: run(lt) for _, lt in rows}
for label, lt in rows:
    print(f"{label:34s}  eval_acc={res[lt]:.4f}")
print(f"\nrecovery perceptron -> tropical_ce_lut: "
      f"{res['tropical_ce_lut'] - res['perceptron']:+.4f}")


## A shift-oriented momentum optimizer: TropicalLion

`TropicalLion` takes the **sign of a momentum**, with power-of-two beta choices
and truncating integer fixed-point state. `momentum_density()` is a diagnostic
for whether that state remains active. Treat this optimizer as an experimental
component and audit it with the rest of the declared recipe.


In [ ]:
m = tt.models.TropicalTransformer(d_model=128, n_heads=4, n_layers=1, d_ff=512,
                                  tau=0.5, max_seq_len=48, task_type="lm", vocab_size=V)
m.loss_type = "tropical_ce"
lion = optim.TropicalLion(m.parameters(), lr_shift=9, grad_bits=16)
gen = torch.Generator().manual_seed(0)
for _ in range(50):
    x, y = batch(24, 48, gen); lion.zero_grad(); m.compute_loss(x, y).backward(); lion.step()
print("Lion momentum density (fraction of live coords):", round(lion.momentum_density(), 3),
      "\n> 0 means a real momentum, not SignSGD-degenerate.")


## Streaming inference: MorphSSM `.step()`

The morphological SSM runs token-by-token with an on-chip ring buffer — the form
the FPGA kernel uses for streaming. Same weights, no recompute.


In [ ]:
ssm = tt.models.MorphSSM(d_model=64, n_layers=2, task_type="lm", vocab_size=V)
from tikitorch.audit import find_closure_violations
print("MorphSSM closure violations (LayerNorm/Linear-head are the declared boundary):",
      [t for _, t in find_closure_violations(ssm)][:4])
print("Swap in TropicalLayerNorm + a tropical head for a fully-closed sequence model.")
